In [1]:
# --- Core Python Libraries ---
import os
import math

# --- Data Processing ---
import numpy as np
import pandas as pd
import h5py

# --- Plotting ---
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.patches import Rectangle

# --- Deep Learning: PyTorch & TorchVision ---
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms, models

# # --- Metrics ---
# from scipy.stats import spearmanr

In [2]:
# !pip install torch torchvision

   ---------------------------------------- 0.0/212.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/212.5 MB 5.6 MB/s eta 0:00:39
   ---------------------------------------- 1.0/212.5 MB 3.0 MB/s eta 0:01:12
   ---------------------------------------- 1.3/212.5 MB 3.2 MB/s eta 0:01:07
   ---------------------------------------- 1.8/212.5 MB 2.5 MB/s eta 0:01:26
   ---------------------------------------- 2.1/212.5 MB 2.6 MB/s eta 0:01:21
   ---------------------------------------- 2.6/212.5 MB 2.2 MB/s eta 0:01:38
    --------------------------------------- 3.1/212.5 MB 2.3 MB/s eta 0:01:31
    --------------------------------------- 3.9/212.5 MB 2.5 MB/s eta 0:01:23
    --------------------------------------- 4.7/212.5 MB 2.6 MB/s eta 0:01:21
    --------------------------------------- 5.2/212.5 MB 2.7 MB/s eta 0:01:17
   - -------------------------------------- 6.0/212.5 MB 2.7 MB/s eta 0:01:17
   - -------------------------------------- 6.8/212.5 MB 2.8 MB/s eta 0

In [13]:
# import torch
# print(torch.__version__)           # e.g., '2.0.0'
# print(torch.version.cuda)

2.7.0+cpu
None


In [7]:
# !conda install pytorch torchvision cpuonly -c pytorch

^C


In [8]:
# !conda install -c conda-forge pytorch-geometric

^C
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: ...working... failed



PackagesNotFoundError: The following packages are not available from current channels:

  - pytorch-geometric

Current channels:

  - https://conda.anaconda.org/conda-forge
  - https://repo.anaconda.com/pkgs/main
  - https://repo.anaconda.com/pkgs/r
  - https://repo.anaconda.com/pkgs/msys2

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




In [3]:
file_path = "elucidata_ai_challenge_data.h5"
with h5py.File(file_path, "r") as f:
        train_images = {k: np.array(v) for k, v in f["images/Train"].items()}
        train_spots  = {k: np.array(v) for k, v in f["spots/Train"].items()}
        test_images  = {k: np.array(v) for k, v in f["images/Test"].items()}
        test_spots   = {k: np.array(v) for k, v in f["spots/Test"].items()}

In [10]:
train_images['S_1'].shape

(2000, 1974, 3)

In [9]:
nb_type = "Train"
if nb_type == "Train":
    patch_w = patch_h = 75

    fig, axes = plt.subplots(len(train_images), 3, figsize=(12, 4*len(train_images)))
    for i, slide in enumerate(train_images):
        img = train_images[slide]
        spots = train_spots[slide]
        # pick a random spot
        idx = np.random.randint(len(spots))
        x0, y0 = spots["x"][idx], spots["y"][idx]
        vals = [spots[f"C{j}"][idx] for j in range(1, 36)]
        
        # compute patch bounds (clamped)
        x1 = max(0, x0 - patch_w//2)
        x2 = min(img.shape[1], x0 + patch_w//2)
        y1 = max(0, y0 - patch_h//2)
        y2 = min(img.shape[0], y0 + patch_h//2)

        # 1) Full slide with red rectangle
        ax = axes[i, 0]
        ax.imshow(img)
        rect = Rectangle((x1, y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.set_title(f"{slide}: spot {idx}")
        ax.axis("off")
        
        # 2) Zoomed patch with a cross at the exact spot
        ax = axes[i, 1]
        patch = img[y1:y2, x1:x2]
        ax.imshow(patch)
        # spot relative to patch
        rel_x = x0 - x1
        rel_y = y0 - y1
        ax.scatter(rel_x, rel_y, marker='x', color='red', s=100, lw=2)
        ax.set_title("zoomed patch")
        ax.axis("off")
        
        # 3) Bar of that spot’s distribution
        ax = axes[i, 2]
        ax.bar(range(1, 36), vals, color=cm.viridis(np.linspace(0, 1, 35)))
        ax.set_xlabel("C1–C35")
        ax.set_ylabel("Abundance")
        ax.set_title("spot composition")

    fig.tight_layout()
    plt.show()

NameError: name 'plt' is not defined